In [3]:
import duckdb

In [4]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [5]:
df = con.execute("""
                 SELECT * 
                 FROM (
                    SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row
                    FROM bronze_z0019
                    WHERE data_ingestao >= '2025-10-15'
                )WHERE row = 1   
                """).fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10002,MARTELO,BT50,100,1500,z0019_1.csv,2025-10-15 00:43:49.890817,1
1,10003,PREGO,BT10,100,60,z0019_2.csv,2025-10-15 00:44:39.134375,1
2,10005,MACHADO,BT50,100,100,z0019_2.csv,2025-10-15 00:44:39.134375,1
3,10004,SERRA,BT50,100,200,z0019_2.csv,2025-10-15 00:44:39.134375,1
4,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2025-10-15 00:43:49.890817,1


In [6]:
df_final = df.drop(columns=['nome_arquivo','data_ingestao','row'])
df_final.rename(columns={
    'NATBR':'id',
    'MAKTX':'nm_produto',
    'WERKS':'id_categoria',
    'MAINS':'id_fornecer',
    'LABST':'vl_preco'
}, inplace=True)

df_final.head(10)

,id,nm_produto,id_categoria,id_fornecer,vl_preco
0,10002,MARTELO,BT50,100,1500
1,10003,PREGO,BT10,100,60
2,10005,MACHADO,BT50,100,100
3,10004,SERRA,BT50,100,200
4,10001,PARAFUSO,BT10,100,100


In [8]:
df_final.dtypes

id              object
nm_produto      object
id_categoria    object
id_fornecer     object
vl_preco        object
dtype: object

In [11]:
df2 = df_final
df2 = df2.astype(
    {
        'id':'int',
        'nm_produto':'str',
        'id_categoria':'str',
        'id_fornecer':'int',
        'vl_preco':'float'
    }
)
df2.dtypes
#df2.head()

id                int64
nm_produto       object
id_categoria     object
id_fornecer       int64
vl_preco        float64
dtype: object

In [12]:
con.execute("""
CREATE TABLE IF NOT EXISTS produtos (
    id BIGINT,
    nm_produto TEXT,
    id_categoria TEXT,
    id_fornecer BIGINT,
    vl_preco FLOAT
    )
""")

In [13]:
df2.head(10)

,id,nm_produto,id_categoria,id_fornecer,vl_preco
0,10002,MARTELO,BT50,100,1500.0
1,10003,PREGO,BT10,100,60.0
2,10005,MACHADO,BT50,100,100.0
3,10004,SERRA,BT50,100,200.0
4,10001,PARAFUSO,BT10,100,100.0


In [15]:
con.execute("INSERT INTO produtos SELECT * from df2")

In [16]:
df_resultado = con.execute("select * from produtos").fetchdf()
df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecer,vl_preco
0,10002,MARTELO,BT50,100,1500.0
1,10003,PREGO,BT10,100,60.0
2,10005,MACHADO,BT50,100,100.0
3,10004,SERRA,BT50,100,200.0
4,10001,PARAFUSO,BT10,100,100.0


In [17]:
con.close()